# RAF-DB dataset exploration and audit
    ## Objective and assumptions
    Classify visible annotations: surprise, fear, disgust, happy, sad, angry, neutral. These labels do not reveal inner feelings.
    Sources: [selected Kaggle mirror](https://www.kaggle.com/datasets/shuvoalok/raf-db-dataset) and
    [original RAF-DB](https://www.whdeng.cn/RAF/model1.html). The mirror states non-commercial research use and restricted redistribution.
    This notebook performs CPU data inspection, not training. Numeric mirror labels require a verified explicit legend;
    the handoff's original label map is initially provisional for this copy.

## Setup and configurable inputs
Prefer authorized files in `data/raw/rafdb`; inspect the existing archive without moving it when that folder is empty.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the FaceAttend project folder.")
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / ".cache/torch")
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from faceattend.utils import environment, load_config, seed_everything, write_json

In [ ]:
import yaml
from faceattend.prepare_data import audit_dataset, make_manifests
from faceattend import CLASS_NAMES, MEAN, STD
config = yaml.safe_load((ROOT / "configs/common.yaml").read_text())
RAW = ROOT / config["raw_root"]
if not RAW.exists():
    RAW = ROOT / "archive (4)"
mapping = json.loads((ROOT / "configs/label_map.json").read_text())
print("Source:", RAW.relative_to(ROOT))
print("Authorized access recorded:", config["authorized_data"])
print("Numeric label legend verified:", config["label_mapping_verified"])

## Audit annotations, images, hashes and source partitions
CSV/folder agreement checks consistency, but does not establish label semantics. The audit reports candidate difference-hash matches; they need human review.

In [ ]:
frame, report = audit_dataset(RAW, ROOT, mapping, config["partition_provenance"])
report["authorized_data"] = config["authorized_data"]
report["label_mapping_verified"] = config["label_mapping_verified"]
write_json(ROOT / "outputs/dataset_audit.json", report)
display(pd.Series({k: v for k, v in report.items() if k not in ["class_counts", "cross_partition_dhash_candidates", "sizes"]}))
display(frame.groupby(["source_partition", "class_name"]).size().unstack(fill_value=0).reindex(columns=CLASS_NAMES))
display(pd.DataFrame(report["sizes"]))
print("Cross-partition near-duplicate candidates:", len(report["cross_partition_dhash_candidates"]))

## Provisional class distribution
Counts use the configured mapping. Interpret class names only after validating the mirror's legend.

In [ ]:
distribution = frame.groupby(["class_name", "source_partition"]).size().unstack(fill_value=0).reindex(CLASS_NAMES)
axis = distribution.plot.bar(figsize=(10, 4), title="Source class counts (configured label map)")
axis.set(xlabel="Visible-expression category", ylabel="Images")
plt.tight_layout()
plt.show()

## Permission and label verification gate
Record authorized use and verified numeric semantics in `configs/common.yaml` before generating training manifests. Do not set these flags merely to bypass an error.

In [ ]:
if not config["authorized_data"]:
    raise RuntimeError("Authorized RAF-DB use has not been recorded; confirm permission before training.")
if not config["label_mapping_verified"]:
    raise RuntimeError("Numeric Kaggle labels are provisional. Verify against original annotations or a trustworthy mirror legend.")
splits = make_manifests(frame, ROOT / config["manifest_root"], seed=config["seed"])
report["manifest_counts"] = {name: len(table) for name, table in splits.items()}
write_json(ROOT / "outputs/dataset_audit.json", report)
display(pd.DataFrame({name: table.class_name.value_counts() for name, table in splits.items()}).fillna(0).reindex(CLASS_NAMES))

## Validate preprocessing using training samples only
These local faces must not be published or retained in shared notebook outputs.

In [ ]:
from PIL import Image
from faceattend.data import make_transform
seed_everything(config["seed"])
sample_path = ROOT / splits["train"].iloc[0].path
image = Image.open(sample_path).convert("RGB")
transform = make_transform(config, training=False)
tensor = transform(image)
assert tensor.shape == (3, 224, 224) and torch.isfinite(tensor).all()
print("RGB evaluation transform passed:", tensor.shape)
# Optional private preview: remove output before sharing this notebook.
SHOW_PRIVATE_SAMPLES = False
if SHOW_PRIVATE_SAMPLES:
    display(image)

## Interpretation
    Preserve source test membership. Train/validation is a seed-42 stratified 85/15 split of source train only.
    Counts matching the benchmark do not authenticate the mirror. Original annotation metadata is absent in this copy,
    identity-level leakage cannot be excluded, and near-duplicate candidates require review.
    Exact path/name/hash overlap causes split creation to stop; do not silently drop or relabel source data.